# Classic SPL weekly gate

Runs before weekly scoring and decides whether this week's run should go ahead.

1. end_date = most recent Saturday before today (US/Central), same rule as Domino. `end_date_override` wins.
2. Skip if `public_results` already has this end_date (week already scored).
3. Check the **source** eltv data (what `copy_base_ltv_by_line` reads), not Fabric's copy, is refreshed through end_date for every line.
4. Outcomes:
   - ready: exits with `run: true`
   - already scored: exits with `run: false` (pipeline succeeds quietly)
   - late: raises, so the pipeline fails and sends the alert


In [ ]:
# Parameters (pipeline overrides these). Must be toggled as the parameter cell.
end_date_override = ""   # "YYYYMMDD" to force a week (skips both checks)
today_override = ""      # "YYYYMMDD" to test as if today were that day
git_checkout = "NB_Classic_SPL_v10.0.0"
model_version = "10.0.0"
ltv_lines = "16,32,72,88,90"

# Source = OneLake shortcut eltv_ltv_base -> Scality tmx-smsiweb/eltv_pipeline/prod/main/core/ltv_base
# Per-line paths: use {line}, e.g. "Files/<shortcut>/ltv_base_{line}"
# One path for all lines: no {line}, and set LINE_COL.
SOURCE_TEMPLATE = "Files/eltv_ltv_base/drv_line_pt={line}"
SOURCE_FORMAT = "parquet"      # auto (delta or parquet), parquet, csv. Source is copied as Binary, so same format as Fabric's copy
LINE_COL = ""               # only used when SOURCE_TEMPLATE has no {line}
DATE_COL = "release_day_pol"


In [ ]:
import json
import datetime
from zoneinfo import ZoneInfo

import notebookutils
from pyspark.sql import functions as F
from pyspark.sql import types as T

import ltv_helpers.fabric_pipeline_helpers as fh
from classic_spl_ltv.config import run_paths

TZ = ZoneInfo("America/Chicago")
END_DATE_WEEKDAY = 5  # Saturday (Python Mon=0)


def last_saturday(today: datetime.date) -> datetime.date:
    """Most recent Saturday strictly before today. Same formula as run_weekly_spl.py."""
    return today + datetime.timedelta(days=(END_DATE_WEEKDAY - today.weekday()) % 7 - 7)


def read_any(path, fmt="auto"):
    if fmt == "csv":
        return spark.read.option("header", True).csv(path)
    if fmt == "parquet":
        return spark.read.parquet(path)
    if notebookutils.fs.exists(f"{path}/_delta_log"):
        return spark.read.format("delta").load(path)
    return spark.read.parquet(path)


def as_date(df, col):
    """Date column whether stored as date, timestamp, or YYYYMMDD / YYYY-MM-DD string or int."""
    dtype = df.schema[col].dataType
    if isinstance(dtype, T.DateType):
        return F.col(col)
    if isinstance(dtype, T.TimestampType):
        return F.to_date(F.col(col))
    s = F.col(col).cast("string")
    return F.coalesce(F.to_date(s, "yyyyMMdd"), F.to_date(s, "yyyy-MM-dd"))


In [ ]:
# Self-check of the date rule
cases = {
    "2026-11-04": "2026-10-31",  # Wed
    "2026-11-06": "2026-10-31",  # Fri
    "2026-11-07": "2026-10-31",  # Sat goes back a full week
    "2026-11-08": "2026-11-07",  # Sun
}
for d, expected in cases.items():
    got = last_saturday(datetime.date.fromisoformat(d))
    assert got == datetime.date.fromisoformat(expected), f"{d}: got {got}, expected {expected}"
print("date rule OK")


## 1. End date

In [ ]:
today = (datetime.datetime.strptime(today_override, "%Y%m%d").date()
         if today_override else datetime.datetime.now(TZ).date())

forced = bool(end_date_override)
end_date = (datetime.datetime.strptime(end_date_override, "%Y%m%d").date()
            if forced else last_saturday(today))

print(f"today={today} ({today:%A})  end_date={end_date}  mode={'override' if forced else 'auto'}")


## 2. Already scored this week?

In [ ]:
p = run_paths.make_paths("score", git_checkout, model_version, end_date=None)

last_published = None
if notebookutils.fs.exists(p.public_results):
    pr = fh.read_delta_table(spark, p.public_results)
    last_published = pr.agg(F.max(as_date(pr, "end_date"))).first()[0]

already_done = last_published is not None and last_published >= end_date
print(f"public_results: {p.public_results}")
print(f"last published end_date={last_published}  already_done={already_done}")


## 3. Source refreshed through end_date? (every line)

In [ ]:
lines = [l.strip() for l in ltv_lines.split(",") if l.strip()]
per_line = {}

if not (forced or already_done):
    assert SOURCE_TEMPLATE and "<shortcut>" not in SOURCE_TEMPLATE, "Set the shortcut name in SOURCE_TEMPLATE"
    if "{line}" in SOURCE_TEMPLATE:
        for line in lines:
            path = SOURCE_TEMPLATE.format(line=line)
            assert notebookutils.fs.exists(path), f"source not found: {path}"
            df = read_any(path, SOURCE_FORMAT)
            per_line[line] = df.agg(F.max(as_date(df, DATE_COL))).first()[0]
    else:
        assert LINE_COL, "SOURCE_TEMPLATE has no {line}; set LINE_COL"
        df = read_any(SOURCE_TEMPLATE, SOURCE_FORMAT)
        rows = (df.withColumn("_line", F.col(LINE_COL).cast("string"))
                  .filter(F.col("_line").isin(lines))
                  .groupBy("_line").agg(F.max(as_date(df, DATE_COL)).alias("mx"))
                  .collect())
        per_line = {line: None for line in lines}
        per_line.update({r["_line"]: r["mx"] for r in rows})

stale = {l: str(d) for l, d in per_line.items() if d is None or d < end_date}
upstream_ready = bool(per_line) and not stale
print(f"max {DATE_COL} by line: { {l: str(d) for l, d in per_line.items()} }")
print(f"upstream_ready={upstream_ready}  stale={stale}")


## 4. Decision

In [ ]:
if forced:
    run, reason = True, "manual end_date override"
elif already_done:
    run, reason = False, "week already scored"
elif upstream_ready:
    run, reason = True, "ready"
else:
    run, reason = False, "late"

result = {
    "run": run,
    "end_date": f"{end_date:%Y%m%d}",
    "reason": reason,
    "already_done": already_done,
    "upstream_ready": upstream_ready,
    "stale_lines": stale,
    "last_published_end_date": str(last_published),
}
print(json.dumps(result, indent=2))

if reason == "late":
    raise RuntimeError(f"eltv source not refreshed through {end_date}. Stale lines: {stale}")

notebookutils.notebook.exit(json.dumps(result))
